## Programa para Adquirir Dados do plano da amostra

In [ ]:
import os
import time
import sys
import numpy as np
import matplotlib.pyplot as plt
import time 
from structured_optics import *
from structured_optics.hologram import slm_hologram
from structured_optics.slmdisplay import My_SLM
from tqdm import tqdm
from camera import *
from utils import *
import h5py

## Funções

In [ ]:
def calculate_fix_std_LG(C, l, p, z):
    return np.sqrt(2*C**2/(2*p + np.abs(l) +1) + 2*np.sqrt(np.abs(C**4/(2*p + np.abs(l) +1)**2 - z**2/beam.k()**2)))/mu

def calculate_fix_std_HG(C, n, z):
    return np.sqrt(2*C**2/(2*n +1) + 2*np.sqrt(np.abs(C**4/(2*n +1)**2 - z**2/beam.k()**2)))/mu

## Parâmetros

In [ ]:
FOLDER = 'Amostras_10_2026/'


#parametros laser e slm
pixel_size = 8e-6  #tamanho do pixel do slm 
lamb = 633e-9   #comprimento de onda
waist = 0.5e-3  #cintura do feixe que sai do slm
x_grating = 3
y_grating = 4

#parametros lentes e propagação
mu = 0.6930012799317185/0.5 #parâmetro de magnificação devido ao sistema 4F após o SLM
z = None #Distância entre o waist pós 4F e a amostra, utilizado para calcular fix std
C = None#0.5e-3  #std a ser fixado


#parametros motor de passo
resolution = 50  #Resolução dos motores de passo
step_size = 200e-9 #tamanho do passo dos motores de passo
NUM_REP = 5  #numero de repetições da medida
z_start = 0.004

#parametros camera
EXPO = None   #exposição camera em segundos
BIT = 8     #bitdepth da camera
pixel_cam = 5.6e-6 #tamanho do pixel da camera
width_cam = 640 #largura da camera
height_cam = 480 #altura da camera
NUM_BACK = 5 #numero de imagens para calcular o background


#parametros amostra
AMOSTRA = "Ferro"   #tipo de amostra
ANGULO_AMOSTRA = 90 #angulo das ranhuras. 90 representa vertical, 45 diagonal e 0 horizontal.



## Primeiro alinhamento e check de equipamentos

In [ ]:
#inicializa camera, slm e lista de modos a serem medidos (até ordem 5)

slm = My_SLM(1)
width = slm.width
height = slm.height

cam = init_camera(bit_depth=BIT, exposure_s = EXPO)


hg_modes = [[0,0], [0,1], [1,0], [2,0], [1,1], [0,2], [3,0], 
            [2,1], [1,2], [0,3], [4,0], [3,1], [2,2], [1,3], 
            [0,4], [5,0], [4,1], [3,2], [2,3], [1,4], [0,5]]
lg_modes = [[0,0], [0,1], [0,-1], [0,2], [1,0], [0,-2], [0,3], 
            [1,1], [1,-1], [0,-3], [0,4], [1, 2], [1,-2], [2, 0], 
            [0,-4], [0,5], [1,3], [1, -3], [2, 1], [2,-1], [0,-5]]

In [ ]:
#inicializa objeto beam que contém os parâmetros do feixe gerado

width = slm.width
height = slm.height

beam = Beam(nix = pixel_size*width/2, Dx = width,
            niy = pixel_size*height/2, Dy = height,
            waist = waist, lamb = lamb)

In [ ]:
#Define qual o campo do objeto beam, gera um holograma a partir desse campo e atualiza a máscara do slm com esse holograma

beam.lg(2,2)
#beam.triangle()
holo = slm_hologram(beam, x_grating, y_grating)
slm.update_image(holo)

## Dados LG waist normal

In [ ]:
beam.waist = waist

with h5py.File(f"data_LG.h5", "w") as file:
    file.attrs.update({
        'expo': f"{EXPO}s",
        'bit_depth': BIT,
        'step_size_nm': int(round(step_size * 1e9)),
        'cam_pixel_size_um': pixel_cam * 1e6,
        'magnification': mu,
        'waist': waist,
        'z': z,
        'lamb': lamb,
    })

    images_ds = file.create_dataset("images", (len(lg_modes), height_cam, width_cam), dtype='uint8')
    index_ds = file.create_dataset("index", (len(lg_modes), 2), dtype='int')

    for i, mode in enumerate(lg_modes):
        l = mode[1]
        p = mode[0]
        index_ds[i] = mode

        beam.lg(l,p)
        holo = slm_hologram(beam, x_grating, y_grating)
        slm.update_image(holo)
        time.sleep(0.2)

        images_ds[i] = take_picture(cam)


## Dados HG waist normal

In [ ]:
beam.waist = waist

with h5py.File(f"data_HG.h5", "w") as file:
    file.attrs.update({
        'expo': f"{EXPO}s",
        'bit_depth': BIT,
        'step_size_nm': int(round(step_size * 1e9)),
        'cam_pixel_size_um': pixel_cam * 1e6,
        'magnification': mu,
        'waist': waist,
        'z': z,
        'lamb': lamb,
    })

    images_ds = file.create_dataset("images", (6, height_cam, width_cam), dtype='uint8')
    index_ds = file.create_dataset("index", (6, 2), dtype='int')

    for n in range(6):
        index_ds[n] = [n,0]

        beam.hg(n,0)
        holo = slm_hologram(beam, x_grating, y_grating)
        slm.update_image(holo)
        time.sleep(0.2)

        images_ds[n] = take_picture(cam)

## Dados LG std fixo

In [ ]:
with h5py.File(f"data_LG_fix.h5", "w") as file:
    file.attrs.update({
        'expo': f"{EXPO}s",
        'bit_depth': BIT,
        'step_size_nm': int(round(step_size * 1e9)),
        'cam_pixel_size_um': pixel_cam * 1e6,
        'magnification': mu,
        'waist': waist,
        'z': z,
        'lamb': lamb,
    })

    images_ds = file.create_dataset("images", (len(lg_modes), height_cam, width_cam), dtype='uint8')
    index_ds = file.create_dataset("index", (len(lg_modes), 2), dtype='int')

    for i, mode in enumerate(lg_modes):
        l = mode[1]
        p = mode[0]
        index_ds[i] = mode

        beam.waist = calculate_fix_std_LG(C, l, p, z)
        beam.lg(l,p)
        holo = slm_hologram(beam, x_grating, y_grating)
        slm.update_image(holo)
        time.sleep(0.2)

        images_ds[i] = take_picture(cam)


## Dados HG std fixo

In [ ]:
with h5py.File(f"data_HG.h5", "w") as file:
    file.attrs.update({
        'expo': f"{EXPO}s",
        'bit_depth': BIT,
        'step_size_nm': int(round(step_size * 1e9)),
        'cam_pixel_size_um': pixel_cam * 1e6,
        'magnification': mu,
        'waist': waist,
        'z': z,
        'lamb': lamb,
    })

    images_ds = file.create_dataset("images", (6, height_cam, width_cam), dtype='uint8')
    index_ds = file.create_dataset("index", (6, 2), dtype='int')

    for n in range(6):
        index_ds[n] = [n,0]

        beam.waist = calculate_fix_std_HG(C, n, z)
        beam.hg(n,0)
        holo = slm_hologram(beam, x_grating, y_grating)
        slm.update_image(holo)
        time.sleep(0.2)

        images_ds[n] = take_picture(cam)